gam

In [ ]:
import datetime
import pandas as pd
import joblib

import sys, os

added_path = os.path.abspath("..")
sys.path.append(added_path)

# utils 모듈 임포트
from utils.naver_searchad_relkeyword import fetch_relkwdstat
from utils.naver_shoppinginsite_search import fetch_category_keyword_data

if added_path in sys.path:
    sys.path.remove(added_path)


# ===== 상수 선언 =====
KEYWORD = "비비고 육개장"
DISCOUNT_PRICE = 2000
PRICE_FACTORS = [0.6, 0.8, 1.0, 1.2, 1.4]
GOOD_ID = "11508428"
CATEGORY_CODE = "50000006"

# ===== 모델 불러오기 =====
gam_package = joblib.load("../models/gam_model.pkl")
gam = gam_package["model"]
X_columns = gam_package["columns"]
goodid_encoder = gam_package["goodid_encoder"]  # 학습 때 저장한 인코더 불러오기

# ===== 예측 데이터 생성 및 수행 =====
rows = []
dates = []

for day_offset in range(0, 7):  # 오늘 포함 7일
    target_date = datetime.date.today() + datetime.timedelta(days=day_offset)
    weekday = pd.Timestamp(target_date).weekday()  # 0=월, 6=일
    dates.append(target_date.strftime("%Y-%m-%d"))

    row = []
    for factor in PRICE_FACTORS:
        price = int(DISCOUNT_PRICE * factor)

        # good_id 문자열을 인코더로 숫자 변환
        good_id_enc = goodid_encoder.transform([GOOD_ID])[0]

        test_data = {
            "discount_price": price,
            "최근4주클릭수_비율": recent_ratio,
            "weekday": weekday,
            "good_id_enc": good_id_enc
        }
        test_df = pd.DataFrame([test_data])
        test_df = test_df.reindex(columns=X_columns, fill_value=0)

        pred_clicks = max(0, gam.predict(test_df)[0])
        row.append(pred_clicks)
    rows.append(row)

# ===== 결과를 DataFrame으로 정리 =====
result_df = pd.DataFrame(rows, index=dates, columns=[f"{int(f*100)}%" for f in PRICE_FACTORS])
print(result_df)

                  60%        80%       100%       120%       140%
2026-04-13  84.580834  74.314989  65.735716  58.585294  52.606000
2026-04-14  77.128990  66.863145  58.283872  51.133450  45.154155
2026-04-15  58.751184  48.485338  39.906066  32.755644  26.776349
2026-04-16  52.962564  42.696719  34.117447  26.967024  20.987730
2026-04-17  33.185210  22.919365  14.340093   7.189670   1.210376
2026-04-18  21.935158  11.669312   3.090040  -4.060382 -10.039677
2026-04-19  57.554716  47.288871  38.709598  31.559176  25.579881


mlp

In [13]:
import datetime
import pandas as pd
import joblib
import torch
import torch.nn as nn

import sys, os

added_path = os.path.abspath("..")
sys.path.append(added_path)

# utils 모듈 임포트
from utils.naver_searchad_relkeyword import fetch_relkwdstat
from utils.naver_shoppinginsite_search import fetch_category_keyword_data

if added_path in sys.path:
    sys.path.remove(added_path)

# ===== 상수 선언 =====
KEYWORD = "참깨라면"
BASE_PRICE = 1000
PRICE_FACTORS = [0.6, 0.8, 1.0, 1.2, 1.4]
pum_id = "A01110"
CATEGORY_CODE = "50000006"

# ===== 모델 불러오기 =====
nn_package = joblib.load("../models/nn_model.pkl")
input_dim = nn_package["input_dim"]
columns = nn_package["columns"]
scaler = nn_package["scaler"]
pumid_encoder = nn_package["pumid_encoder"]   # 학습 때 저장한 인코더 불러오기

class SimpleMLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(SimpleMLPRegressor, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleMLPRegressor(input_dim=input_dim)
model.load_state_dict(nn_package["model_state"])
model.eval()

# ===== 예측 수행 =====
rows = []
dates = []

for day_offset in range(0, 7):
    target_date = datetime.date.today() + datetime.timedelta(days=day_offset)
    weekday = target_date.weekday()
    dates.append(target_date.strftime("%Y-%m-%d"))

    row = []
    for factor in PRICE_FACTORS:
        price = int(BASE_PRICE * factor)

        # pum_id 문자열을 인코더로 숫자 변환
        pum_id_enc = pumid_encoder.transform([pum_id])[0]

        test_data = {
            "1개당가격": price,
            "최근4주클릭수_비율": recent_ratio,
            "weekday": weekday,
            "pum_id_enc": pum_id_enc
        }
        test_df = pd.DataFrame([test_data])
        test_df = test_df.reindex(columns=columns, fill_value=0)

        # 스케일링
        X_scaled = scaler.transform(test_df)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

        with torch.no_grad():
            pred_clicks = model(X_tensor).cpu().numpy()[0][0]

        row.append(pred_clicks)
    rows.append(row)

# ===== 결과를 DataFrame으로 정리 =====
result_df = pd.DataFrame(rows, index=dates, columns=[f"{int(f*100)}%" for f in PRICE_FACTORS])
print(result_df)

                  60%        80%       100%       120%       140%
2026-04-13  18.616455  15.902265  13.188054  10.473849   7.759645
2026-04-14  31.124352  33.854778  33.453732  27.104349  24.390148
2026-04-15  26.164379  28.298487  30.541218  32.783947  35.100948
2026-04-16  22.457270  24.501003  26.634842  28.877573  31.120304
2026-04-17  18.750160  20.793894  22.837629  24.971197  27.213928
2026-04-18  15.392252  17.306288  19.329853  21.353418  23.466549
2026-04-19  12.704340  14.527835  16.482941  18.506504  20.530069
